In [2]:
import pandas as pd

df_finance = pd.read_csv('../data/raw/Expenses_clean.csv')
print(df_finance.shape)
print(df_finance.dtypes)
print(df_finance.head())

(938, 6)
date_time        str
category         str
account          str
amount       float64
currency         str
tags             str
dtype: object
             date_time          category account  amount currency   tags
0  2025-11-30 00:00:00            Health  acct_1   114.0      BYN  tag_1
1  2025-11-29 00:00:00              Food  acct_1     5.0      BYN  tag_1
2  2025-11-27 00:00:00  Public transport  acct_2     1.0      BYN  tag_1
3  2025-11-27 00:00:00              Cafe  acct_1    10.0      BYN  tag_1
4  2025-11-27 00:00:00  Public transport  acct_2     1.0      BYN  tag_1


In [3]:
df_anomaly = pd.read_csv('../data/raw/bank_transactions_data_2.csv')
print(df_anomaly.shape)
print(df_anomaly.dtypes)
print(df_anomaly.head())

(2512, 16)
TransactionID                  str
AccountID                      str
TransactionAmount          float64
TransactionDate                str
TransactionType                str
Location                       str
DeviceID                       str
IP Address                     str
MerchantID                     str
Channel                        str
CustomerAge                  int64
CustomerOccupation             str
TransactionDuration          int64
LoginAttempts                int64
AccountBalance             float64
PreviousTransactionDate        str
dtype: object
  TransactionID AccountID  TransactionAmount      TransactionDate  \
0      TX000001   AC00128              14.09  2023-04-11 16:29:14   
1      TX000002   AC00455             376.24  2023-06-27 16:44:19   
2      TX000003   AC00019             126.29  2023-07-10 18:16:08   
3      TX000004   AC00070             184.50  2023-05-05 16:32:11   
4      TX000005   AC00411              13.45  2023-10-16 17:51:24   



In [5]:
import pandas as pd

# --- Finance Dataset ---
df_finance = pd.read_csv('../data/raw/Expenses_clean.csv')

# Parse date, drop useless columns
df_finance['date_time'] = pd.to_datetime(df_finance['date_time'])
df_finance = df_finance.drop(columns=['tags', 'currency', 'account'])

# Add useful time columns
df_finance['month'] = df_finance['date_time'].dt.month
df_finance['day_of_week'] = df_finance['date_time'].dt.dayofweek
df_finance['month_year'] = df_finance['date_time'].dt.to_period('M')

print("Finance clean:")
print(df_finance.head())
print(df_finance['category'].value_counts())

# --- Anomaly Dataset ---
df_anomaly = pd.read_csv('../data/raw/bank_transactions_data_2.csv')  # update filename if different

# Drop low-signal columns
df_anomaly = df_anomaly.drop(columns=['DeviceID', 'IP Address', 'MerchantID', 
                                       'CustomerAge', 'CustomerOccupation', 
                                       'TransactionID', 'AccountID', 'Location'])

# Parse dates
df_anomaly['TransactionDate'] = pd.to_datetime(df_anomaly['TransactionDate'])
df_anomaly['PreviousTransactionDate'] = pd.to_datetime(df_anomaly['PreviousTransactionDate'])

# Engineer useful features
df_anomaly['hour'] = df_anomaly['TransactionDate'].dt.hour
df_anomaly['day_of_week'] = df_anomaly['TransactionDate'].dt.dayofweek
df_anomaly['days_since_last'] = (df_anomaly['TransactionDate'] - df_anomaly['PreviousTransactionDate']).dt.days.abs()

print("\nAnomaly clean:")
print(df_anomaly.head())
print(df_anomaly.dtypes)

Finance clean:
   date_time          category  amount  month  day_of_week month_year
0 2025-11-30            Health   114.0     11            6    2025-11
1 2025-11-29              Food     5.0     11            5    2025-11
2 2025-11-27  Public transport     1.0     11            3    2025-11
3 2025-11-27              Cafe    10.0     11            3    2025-11
4 2025-11-27  Public transport     1.0     11            3    2025-11
category
Public transport     414
Cafe                 169
Food                 142
Taxi                  62
Gifts                 41
Leisure               30
Other                 24
Health                22
Loan given            11
University             9
Job                    5
Bought for myself      4
Clothes                3
Fines                  2
Name: count, dtype: int64

Anomaly clean:
   TransactionAmount     TransactionDate TransactionType Channel  \
0              14.09 2023-04-11 16:29:14           Debit     ATM   
1             376.24 2023-06

In [6]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import LabelEncoder
import numpy as np

# Encode categorical columns
df_model = df_anomaly.copy()

le_type = LabelEncoder()
le_channel = LabelEncoder()
df_model['TransactionType_enc'] = le_type.fit_transform(df_model['TransactionType'])
df_model['Channel_enc'] = le_channel.fit_transform(df_model['Channel'])

# Select features for the model
features = [
    'TransactionAmount',
    'TransactionDuration',
    'LoginAttempts',
    'AccountBalance',
    'hour',
    'day_of_week',
    'days_since_last',
    'TransactionType_enc',
    'Channel_enc'
]

X = df_model[features]

# Train Isolation Forest
model = IsolationForest(contamination=0.05, random_state=42)
df_model['anomaly'] = model.fit_predict(X)
df_model['anomaly_score'] = model.decision_function(X)

# -1 = anomaly, 1 = normal
anomalies = df_model[df_model['anomaly'] == -1]
print(f"Total transactions: {len(df_model)}")
print(f"Anomalies detected: {len(anomalies)}")
print(f"\nSample anomalies:")
print(anomalies[['TransactionAmount', 'AccountBalance', 'LoginAttempts', 
                  'TransactionDuration', 'hour', 'anomaly_score']].head(10))

Total transactions: 2512
Anomalies detected: 126

Sample anomalies:
     TransactionAmount  AccountBalance  LoginAttempts  TransactionDuration  \
26              246.93          673.35              5                  158   
42              241.42        13357.02              1                  116   
85             1340.19         8654.28              1                   30   
88                9.61        12883.91              1                  179   
117              56.98         5688.37              2                  122   
147             514.95          421.93              5                  142   
176            1362.55         2429.68              1                  106   
193             280.57          293.24              1                   24   
218             890.24         5140.93              1                  161   
231             705.60        14804.70              3                  124   

     hour  anomaly_score  
26     16      -0.016086  
42     18      -0.0

In [7]:
def explain_anomaly(row):
    reasons = []
    
    if row['LoginAttempts'] >= 3:
        reasons.append(f"High login attempts ({int(row['LoginAttempts'])})")
    if row['TransactionAmount'] > df_model['TransactionAmount'].quantile(0.95):
        reasons.append(f"Unusually large amount (${row['TransactionAmount']:.2f})")
    if row['TransactionAmount'] > row['AccountBalance'] * 0.5:
        reasons.append(f"Amount is over 50% of account balance")
    if row['hour'] < 6 or row['hour'] > 22:
        reasons.append(f"Transaction at unusual hour ({int(row['hour'])}:00)")
    if row['TransactionDuration'] > df_model['TransactionDuration'].quantile(0.95):
        reasons.append(f"Unusually long transaction duration")
    
    return reasons if reasons else ["Statistical outlier across multiple factors"]

anomalies['reasons'] = anomalies.apply(explain_anomaly, axis=1)

print("Anomalies with explanations:")
for _, row in anomalies.head(8).iterrows():
    print(f"\n Amount: ${row['TransactionAmount']:.2f} | "
          f"Balance: ${row['AccountBalance']:.2f} | "
          f"Login attempts: {int(row['LoginAttempts'])}")
    print(f"  Reasons: {row['reasons']}")

Anomalies with explanations:

 Amount: $246.93 | Balance: $673.35 | Login attempts: 5
  Reasons: ['High login attempts (5)']

 Amount: $241.42 | Balance: $13357.02 | Login attempts: 1
  Reasons: ['Statistical outlier across multiple factors']

 Amount: $1340.19 | Balance: $8654.28 | Login attempts: 1
  Reasons: ['Unusually large amount ($1340.19)']

 Amount: $9.61 | Balance: $12883.91 | Login attempts: 1
  Reasons: ['Statistical outlier across multiple factors']

 Amount: $56.98 | Balance: $5688.37 | Login attempts: 2
  Reasons: ['Statistical outlier across multiple factors']

 Amount: $514.95 | Balance: $421.93 | Login attempts: 5
  Reasons: ['High login attempts (5)', 'Amount is over 50% of account balance']

 Amount: $1362.55 | Balance: $2429.68 | Login attempts: 1
  Reasons: ['Unusually large amount ($1362.55)', 'Amount is over 50% of account balance']

 Amount: $280.57 | Balance: $293.24 | Login attempts: 1
  Reasons: ['Amount is over 50% of account balance']


In [8]:
import os

os.makedirs('../data/processed', exist_ok=True)

# Save cleaned finance data
df_finance.to_csv('../data/processed/finance_clean.csv', index=False)

# Save anomaly results with scores and reasons
df_model['reasons'] = df_model.apply(
    lambda row: explain_anomaly(row) if row['anomaly'] == -1 else [], axis=1
)
df_model.to_csv('../data/processed/anomaly_results.csv', index=False)

print("Saved:")
print(f"  finance_clean.csv — {len(df_finance)} rows")
print(f"  anomaly_results.csv — {len(df_model)} rows ({len(anomalies)} flagged)")

Saved:
  finance_clean.csv — 938 rows
  anomaly_results.csv — 2512 rows (126 flagged)
